In [ ]:
!pip install -q openai datasets scikit-learn tqdm pandas

In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENAI_API_KEY")

    if not API_KEY:
        raise ValueError("Key not found.")

    os.environ["OPENAI_API_KEY"] = API_KEY

except Exception as err:
    print({err})
    raise

try:
    client = OpenAI(api_key=API_KEY)

    test = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": "What is the climate in Norway?"}],
    )
    print(test.choices[0].message.content)
    print(f"Tokens: {test.usage.total_tokens}")

except Exception as err:
    print({err})
    raise

In [ ]:
import pandas as pd
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

dataset_original = load_dataset("ailsntua/QEvasion")
test_df = dataset_original['test'].to_pandas()

response_to_label_map = {
    "explicit": "Explicit",
    "implicit": "Implicit",
    "dodging": "Dodging",
    "general": "General",
    "deflection": "Deflection",
    "partial": "Partial/half-answer",
    "partial/half-answer": "Partial/half-answer",
    "partial/half answer": "Partial/half-answer",
    "half-answer": "Partial/half-answer",
    "half answer": "Partial/half-answer",
    "declining": "Declining to answer",
    "declining to answer": "Declining to answer",
    "ignorance": "Claims ignorance",
    "claims ignorance": "Claims ignorance",
    "clarification": "Clarification",
}

In [ ]:
TAXONOMY_TEXT = """
1. Explicit - The information requested is explicitly stated
2. Implicit - The information requested is given, but without being explicitly stated
3. Dodging - Ignoring the question altogether
4. General - The information provided is too general/lacks the requested specificity
5. Deflection - Starts on topic but shifts the focus
6. Partial/half-answer - Offers only a specific component of the requested information
7. Declining to answer - Acknowledge the question but directly or indirectly refusing to answer
8. Claims ignorance - The answerer claims/admits not to know the answer
9. Clarification - Does not provide the requested information and asks for clarification
"""

SYSTEM_MESSAGE = f"""You are an expert in analyzing political discourse and question evasion patterns.
Your task is to classify the type of response provided by an interviewee using the following taxonomy:

{TAXONOMY_TEXT}

In the following examples, you will see 9 demonstration cases (one for each taxonomy class).
Each demonstration consists of 2 messages: one user message with the question and answer, followed by one assistant message showing the step-by-step reasoning and final classification."""

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": """### Question ###
Do you have your own views about PR at Westminster don't you?

### Answer ###
I do."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Do you have your own views about PR at Westminster?"
2. The answer is: "I do."
3. This directly confirms having views, explicitly answering the yes/no question.
4. The information requested is explicitly stated.

Taxonomy code: Explicit"""
    },

    {
        "role": "user",
        "content": """### Question ###
Are you going to watch television?

### Answer ###
What else is there to do?"""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Are you going to watch television?"
2. The answer is: "What else is there to do?"
3. This rhetorical question suggests they will watch TV (since there's nothing else to do).
4. The information is given indirectly, not explicitly stated.

Taxonomy code: Implicit"""
    },

    {
        "role": "user",
        "content": """### Question ###
Do you like my new dress?

### Answer ###
We are late."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Do you like my new dress?"
2. The answer is: "We are late."
3. The interviewee completely ignores the question about the dress.
4. They change the subject entirely to being late.
5. This is ignoring the question altogether.

Taxonomy code: Dodging"""
    },

    {
        "role": "user",
        "content": """### Question ###
Did you eat the last piece of pie?

### Answer ###
I have to admit that this was a great recipe, I always like it when there are chocolate chips in the dough."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Did you eat the last piece of pie?"
2. The answer acknowledges the pie ("great recipe") but then shifts focus to chocolate chips.
3. The interviewee starts on topic but goes on a tangent without answering whether they ate it.
4. This is deflection - acknowledging the question but shifting focus.

Taxonomy code: Deflection"""
    },

    {
        "role": "user",
        "content": """### Question ###
Did you enjoy the film?

### Answer ###
The directing was great."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Did you enjoy the film?"
2. The answer only addresses the directing, which is one component of a film.
3. The interviewee doesn't address acting, plot, cinematography, or overall enjoyment.
4. This is a partial answer - only addressing one specific component.

Taxonomy code: Partial/half-answer"""
    },

    {
        "role": "user",
        "content": """### Question ###
What's your favorite film?

### Answer ###
Fight Club, Filth, and Hereditary."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for ONE favorite film (singular).
2. The answer provides THREE films.
3. This makes the desired information unclear - which one is truly the favorite?
4. The reply is too general and lacks the requested specificity.

Taxonomy code: General"""
    },

    {
        "role": "user",
        "content": """### Question ###
The hypothesis I was discussing, wouldn't you regard that as a defeat?

### Answer ###
I am not going to prophesy what will happen."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for their view on whether something would be a defeat.
2. The answer is: "I am not going to prophesy what will happen."
3. The interviewee directly states they won't answer.
4. They acknowledge the question but refuse to provide the information.

Taxonomy code: Declining to answer"""
    },

    {
        "role": "user",
        "content": """### Question ###
On what precise date did the government order the refit of the HMAS Kanimbla in preparation for its forward deployment to a possible war against Iraq?

### Answer ###
I do not know that date. I will find out and let the House know."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for a specific date.
2. The answer is: "I do not know that date."
3. The interviewee admits they don't have the information.
4. This is claiming ignorance - they state they don't know the answer.

Taxonomy code: Claims ignorance"""
    },

    {
        "role": "user",
        "content": """### Question ###
Was it your decision to release the fund?

### Answer ###
You mean the public fund?"""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Was it your decision to release the fund?"
2. The answer is: "You mean the public fund?"
3. The interviewee doesn't provide any information about the decision.
4. Instead, they ask for clarification about which fund.
5. This is asking for clarification without giving the requested data.

Taxonomy code: Clarification"""
    }
]

USER_PROMPT_TEMPLATE = """Now, you have to analyze this new case:

### Question ###
{question}

### Answer ###
{interview_answer}

Instructions for your analysis:
1. Think step by step about the relationship between the question and the answer.
2. Provide a brief explanation of your reasoning.
3. You MUST end your response with "Taxonomy code: [Label Name]" exactly as shown in the examples above.

Let's think step by step and provide your classification."""

In [ ]:
from tqdm import tqdm
import time

predictions = []
api_call_times = []

print(f"Starting GPT-5 prompting on {len(test_df)} test examples...\n")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing"):
    question = row['question']
    interview_answer = row['interview_answer']

    user_prompt = USER_PROMPT_TEMPLATE.format(
        interview_answer=interview_answer,
        question=question
    )

    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE}
    ] + FEW_SHOT_EXAMPLES + [
        {"role": "user", "content": user_prompt}
    ]

    start_time = time.time()

    try:
        response = client.chat.completions.create(
            model="gpt-5",
            messages=messages,
        )

        api_call_times.append(time.time() - start_time)
        response_text = response.choices[0].message.content.strip()

        predicted_label = None
        if "taxonomy code:" in response_text.lower():
            parts = response_text.lower().split("taxonomy code:")
            if len(parts) > 1:
                candidate = parts[-1].strip().split('\n')[0].strip()
                predicted_label = response_to_label_map.get(candidate)

        if predicted_label is None:
            print(f"\nError at index {idx}: Label not found in response")
            break

        predictions.append(predicted_label)

    except Exception as e:
        print(f"\nError at index {idx}: {e}")
        break

print(f"\n Prompting complete: {len(predictions)} predictions")
print(f"  Total time: {sum(api_call_times)/60:.1f} minutes\n")

results_df = test_df.copy()
results_df['predicted_label'] = predictions

output_file = "gpt5-fscot-test-predictions.csv"
results_df.to_csv(output_file, index=False)

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support
import numpy as np
import pandas as pd

all_labels = [
    "Clarification",
    "Claims ignorance",
    "Declining to answer",
    "Deflection",
    "Dodging",
    "Explicit",
    "General",
    "Implicit",
    "Partial/half-answer"
]

label_to_idx = {lbl: i for i, lbl in enumerate(all_labels)}
n = len(results_df)
y_true_binary = np.zeros((n, len(all_labels)), dtype=int)
y_pred_binary = np.zeros((n, len(all_labels)), dtype=int)

for idx, ((_, row), pred) in enumerate(zip(results_df.iterrows(), predictions)):
    ann_set = {row['annotator1'], row['annotator2'], row['annotator3']}
    for lbl in ann_set:
        if lbl in label_to_idx:
            y_true_binary[idx, label_to_idx[lbl]] = 1
    if pred in label_to_idx:
        y_pred_binary[idx, label_to_idx[pred]] = 1

precisions, recalls, f1s, supports = precision_recall_fscore_support(
    y_true_binary, y_pred_binary, average=None, labels=range(len(all_labels)), zero_division=0
)

print("Subtask 2: Evasion Classification\n")

print(f"Macro F1 (multi-truth label union): {np.mean(f1s):.4f}  Micro F1: {f1_score(y_true_binary, y_pred_binary, average='micro'):.4f}\n")

print("Per-class metrics:")
for i, lbl in enumerate(all_labels):
    print(f"{lbl:<25s} P={precisions[i]:.2f} R={recalls[i]:.2f} F1={f1s[i]:.2f} Support={supports[i]:d}")



Subtask 2: Evasion Classification

Macro F1 (multi-truth label union): 0.4564  Micro F1: 0.4038

Per-class metrics:
Clarification             P=1.00 R=1.00 F1=1.00 Support=4
Claims ignorance          P=0.80 R=0.27 F1=0.40 Support=15
Declining to answer       P=0.67 R=0.94 F1=0.78 Support=17
Deflection                P=0.31 R=0.78 F1=0.44 Support=51
Dodging                   P=0.82 R=0.09 F1=0.17 Support=96
Explicit                  P=0.76 R=0.59 F1=0.67 Support=115
General                   P=0.54 R=0.06 F1=0.11 Support=113
Implicit                  P=0.77 R=0.17 F1=0.28 Support=99
Partial/half-answer       P=0.33 R=0.21 F1=0.26 Support=14


In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

CLARITY_MAPPING = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "Dodging": "Ambivalent",
    "General": "Ambivalent",
    "Deflection": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply",
}

print("Subtask 1: Evasion-based Clarity Classification\n")


y_true_clarity = results_df['clarity_label'].tolist()
y_pred_clarity = [CLARITY_MAPPING[pred] for pred in predictions]

accuracy_clarity = sum(1 for yt, yp in zip(y_true_clarity, y_pred_clarity) if yt == yp) / len(y_true_clarity)

clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
macro_f1_clarity = f1_score(y_true_clarity, y_pred_clarity, average='macro', labels=clarity_labels, zero_division=0)

print(f"Accuracy: {accuracy_clarity:.4f}")
print(f"Macro F1: {macro_f1_clarity:.4f}")

print("\nPer-class metrics:")
print(classification_report(y_true_clarity, y_pred_clarity, labels=clarity_labels, zero_division=0))

Subtask 1: Evasion-based Clarity Classification

Accuracy: 0.7532
Macro F1: 0.7109

Per-class metrics:
                 precision    recall  f1-score   support

    Clear Reply       0.61      0.68      0.64        79
     Ambivalent       0.85      0.77      0.81       206
Clear Non-Reply       0.58      0.83      0.68        23

       accuracy                           0.75       308
      macro avg       0.68      0.76      0.71       308
   weighted avg       0.77      0.75      0.76       308

